# 這是一份完整的基於對單個因子的驗證
使用duckdb對奇怪因子做驗證

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
# 暫時禁用future warningq
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
# 專案根目錄（alphagen-master/alphagen-master）
# Jupyter cwd 在 testing/ 時用 parent；否則用 cwd
_cwd = Path.cwd()
root = _cwd.parent if _cwd.name == "testing" else _cwd
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"Project root: {root}")

In [ ]:
import polars as pl
from src.factor_engine import FactorEngine

RAM_PATH = "/dev/shm/qlib_ram"
start_time = "2015-06-01"
end_time = "2026-05-29"

# 參考 factor_test copy.ipynb 建立 FactorEngine；此處只取 open 的完整索引
engine = FactorEngine(
    qlib_data_path=RAM_PATH,
    instrument="all",
    start_time=start_time,
    end_time=end_time,
    device="cuda:0",
    max_backtrack_days=0,
    max_future_days=0,
    features=["open"],
    adjust_factor="none",
)

engine_index = engine.calc_df("$open", col_name="_open").index.to_frame(index=False)
engine_index.columns = ["date", "symbol"]
engine_pairs = (
    pl.from_pandas(engine_index)
    .with_columns(
        pl.col("date").cast(pl.Date),
        pl.col("symbol").cast(pl.String),
    )
    .sort(["symbol", "date"])
)

del engine_index
print(f"FactorEngine 基準：{engine_pairs.height:,} 個 date/symbol pairs")
engine_pairs.head()

In [ ]:
import duckdb

# 讀取 DuckDB，再以 FactorEngine 的完整 date/symbol 面板為準對齊
duckdb_path = "/home/newt/quant_project/duckdb/daily_data.duckdb"
con = duckdb.connect(database=duckdb_path, read_only=True)
duck_df = (
    con.execute("SELECT * FROM daily_data").pl()
    .with_columns(
        pl.col("date").cast(pl.Date),
        pl.col("symbol").cast(pl.String),
        pl.lit(True).alias("_duck_present"),
    )
)

# Qlib/Engine 是主索引；基準外的 DuckDB 資料只報告、不併入
outside_engine = duck_df.select(["date", "symbol"]).join(
    engine_pairs, on=["date", "symbol"], how="anti"
)
if outside_engine.height:
    print(f"注意：DuckDB 有 {outside_engine.height:,} 個 pairs 不在 Qlib/Engine 基準內，將不納入")

df = (
    engine_pairs.join(duck_df, on=["date", "symbol"], how="left")
    .sort(["symbol", "date"])
)
missing_pair_count = df["_duck_present"].null_count()

# OHLC 沿用同一股票前一個交易日；其餘未指定欄位保留 null
price_cols = ["open", "high", "low", "close"]
df = (
    df.with_columns(
        [pl.col(col).forward_fill().over("symbol").alias(col) for col in price_cols]
    )
    .with_columns(
        pl.col("volume").fill_null(0),
        pl.col("return_day").fill_null(0),
    )
    .drop("_duck_present")
)

print(f"DuckDB 原始：{duck_df.height:,} rows")
print(f"Engine 對齊後：{df.height:,} rows（新增並填充 {missing_pair_count:,} 個 pairs）")
df.select(["date", "symbol", *price_cols, "volume", "return_day"]).head()

In [ ]:
price_cols = ['open', 'high', 'low', 'close']
adj_col = 'adj_factor'

# 原始復權因子必須為正；engine 補出的日期其 adj_factor 為 null，計算時視為 1
invalid_adj_count = df.filter(
    pl.col(adj_col).is_not_null() & (pl.col(adj_col) <= 0)
).height
if invalid_adj_count:
    raise ValueError(f"adj_factor 有 {invalid_adj_count:,} 筆小於或等於 0，復權無法進行")

# 每檔股票按日期由舊到新累乘當日復權因子
df = (
    df.sort(['symbol', 'date'])
    .with_columns(
        pl.col(adj_col)
        .fill_null(1.0)
        .cum_prod()
        .over('symbol')
        .alias('cum_adj_factor')
    )
    .with_columns(
        [
            (pl.col(col) * pl.col('cum_adj_factor')).alias(f'{col}_adj')
            for col in price_cols
        ]
    )
)

df.select(['symbol', 'date', adj_col, 'cum_adj_factor', *[f'{col}_adj' for col in price_cols]]).head()

In [ ]:
from src.factor.single_factor_eval import prepare_eval_returns

# 通用未來報酬 / lag 報酬寫進原始 df，驗證時直接用
df = prepare_eval_returns(df, price_col="close_adj")
df.select(
    ["symbol", "date", "close_adj", "1day_return", "3day_return", "returns_lag0", "returns_lag1"]
).head()

In [ ]:
# 計算 'return_day' 過去22/66/126天的標準差
import polars as pl

window_sizes = [22, 66, 126]
for w in window_sizes:
    df = df.with_columns(
        (-pl.col("return_day").rolling_std(window_size=w))
        .over("symbol")
        .alias(f"return_day_std_{w}")
    )

df.select([f"return_day_std_{w}" for w in window_sizes]).head()

In [ ]:
# 下行波動率：每個視窗只採用負收益計算標準差
window_sizes = [22, 66, 126]
factor_names = [f"vol_down_std_{w}d" for w in window_sizes]
down_return = pl.when(pl.col("return_day") < 0).then(pl.col("return_day"))

df = df.with_columns([
    pl.when(
        down_return.is_not_null()
        .cast(pl.UInt16)
        .rolling_sum(window_size=w, min_samples=1)
        .over("symbol")
        == 1
    )
    .then(0.0)
    .otherwise(
        down_return.rolling_std(window_size=w, min_samples=1).over("symbol")
    )
    .alias(factor_name)
    for w, factor_name in zip(window_sizes, factor_names)
])

df.select(factor_names).head()

In [ ]:
from src.factor.single_factor_eval import run_single_factor_test

alpha = run_single_factor_test(df, factor_name="return_day_std_22", group_count=5)